In [1]:
import time
import re
from datetime import datetime
from typing import List, Dict, Optional
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup


class DiadoraCompetitiveAnalyzer:
    """Analizzatore completo con auto-detection selettori integrata"""

    def __init__(self, headless: bool = False, auto_detect: bool = True):
        self.headless = headless
        self.auto_detect = auto_detect
        self.driver = None
        self.selectors = None
        self.all_products = []

        print(self._get_banner())
        self.setup_driver()

    def _get_banner(self):
        return """
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║         DIADORA COMPETITIVE ANALYSIS - VERSIONE INTEGRATA v3.0               ║
║                                                                              ║
║              Auto-Detection Selettori + Analisi Completa                     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

    def setup_driver(self):
        print("\n" + "=" * 80)
        print("🔧 CONFIGURAZIONE CHROME WEBDRIVER")
        print("=" * 80)

        options = Options()

        if self.headless:
            options.add_argument('--headless')
            print("📱 Modalità: Headless (background)")
        else:
            print("👁️  Modalità: Visibile (vedrai il browser)")

        options.add_argument('--window-size=1920,1200')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('user-agent=Mozilla/5.0')

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        self.driver.implicitly_wait(10)

        print("✅ WebDriver configurato con successo!\n")

    def navigate_and_prepare(self):
        print("=" * 80)
        print("📍 NAVIGAZIONE SITO DIADORA")
        print("=" * 80)

        print("\n1️⃣ Apertura homepage...")
        self.driver.get("https://www.diadora.com/it/it/")
        time.sleep(3)

        print("2️⃣ Gestione cookie consent...")
        self._handle_cookies()

        print("3️⃣ Navigazione sezione scarpe...")
        self._navigate_to_shoes()

        print("4️⃣ Caricamento prodotti (scroll)...")
        self._scroll_page()

        print("\n✅ Pagina pronta per l'analisi!\n")

    def _handle_cookies(self):
        cookie_selectors = [
            (By.ID, "onetrust-accept-btn-handler"),
            (By.XPATH, "//button[contains(text(), 'Accetta')]"),
            (By.XPATH, "//button[contains(text(), 'Accept')]"),
        ]

        for by, selector in cookie_selectors:
            try:
                btn = WebDriverWait(self.driver, 3).until(
                    EC.element_to_be_clickable((by, selector))
                )
                btn.click()
                print("   ✅ Cookie accettati")
                return
            except:
                continue

        print("   ⚠️ Cookie banner non trovato")

    def _navigate_to_shoes(self):
        """Naviga alla sezione scarpe in modo robusto"""

        print("   🔍 Ricerca link 'Scarpe'...")

        shoe_links = [
            (By.XPATH, "//a[contains(text(), 'Scarpe')]"),
            (By.XPATH, "//a[contains(text(), 'SCARPE')]"),
            (By.XPATH, "//a[contains(@href, '/scarpe')]"),
            (By.XPATH, "//a[contains(@href, '/shoes')]"),
        ]

        for by, selector in shoe_links:
            try:
                link = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((by, selector))
                )

                if not link.is_displayed():
                    continue

                print(f"   ✅ Trovato link visibile: '{link.text}'")

                self.driver.execute_script("arguments[0].scrollIntoView(true);", link)
                time.sleep(0.5)

                try:
                    WebDriverWait(self.driver, 5).until(
                        EC.element_to_be_clickable((by, selector))
                    )
                    link.click()
                    time.sleep(2)
                    return
                except:
                    pass

                print("   ⚠️ Click normale fallito → uso JavaScript click")
                self.driver.execute_script("arguments[0].click();", link)
                time.sleep(2)
                return

            except:
                continue

        print("\n   ⚠️ Link 'Scarpe' non trovato automaticamente")
        if not self.headless:
            print("   👉 Clicca manualmente su 'SCARPE' nel browser")
            input("   Premi INVIO per continuare... ")

    def _scroll_page(self):
        last_height = self.driver.execute_script("return document.body.scrollHeight")

        for _ in range(10):
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = self.driver.execute_script("return document.body.scrollHeight")

            if new_height == last_height:
                break
            last_height = new_height

        print("   ✅ Scroll completato")

    def auto_detect_selectors(self):
        print("=" * 80)
        print("🔍 AUTO-DETECTION SELETTORI CSS")
        print("=" * 80)

        soup = BeautifulSoup(self.driver.page_source, 'html.parser')

        with open('diadora_analysis.html', 'w', encoding='utf-8') as f:
            f.write(soup.prettify())

        print("💾 HTML salvato in diadora_analysis.html")

        selectors = {}

        container_info = self._find_product_container(soup)

        if container_info:
            selectors['container_tag'] = container_info['tag']
            selectors['container_class'] = container_info['class']
            print(f"   ✅ Container rilevato: {container_info['tag']}.{container_info['class']}")
        else:
            print("   ❌ Nessun container rilevato, uso fallback")
            selectors = self._get_fallback_selectors()

        self.selectors = selectors
        return selectors

    def _find_product_container(self, soup: BeautifulSoup) -> Optional[Dict]:
        """Trova il container che racchiude i prodotti"""

        patterns = [
            ('div', r'product-grid__item', 5),
            ('div', r'product-tile', 5),
            ('div', r'product.*tile', 5),
            ('div', r'product.*item', 5),
            ('li', r'product', 5),
        ]

        candidates = []

        for tag, class_pattern, min_count in patterns:
            elements = soup.find_all(tag, class_=re.compile(class_pattern, re.I))

            if len(elements) >= min_count:
                first_elem = elements[0]
                classes = first_elem.get('class', [])

                if classes:
                    candidates.append({
                        'tag': tag,
                        'class': classes[0],
                        'count': len(elements)
                    })

        if candidates:
            return max(candidates, key=lambda x: x['count'])

        return None

    def _get_fallback_selectors(self):
        return {
            'container_tag': 'div',
            'container_class': 'product-grid__item'
        }

    def scrape_products(self, max_products: int = 200):
        print("\n" + "=" * 80)
        print("📦 SCRAPING PRODOTTI DIADORA")
        print("=" * 80)

        if not self.selectors:
            self.auto_detect_selectors()

        soup = BeautifulSoup(self.driver.page_source, 'html.parser')

        containers = soup.find_all(
            self.selectors['container_tag'],
            class_=self.selectors['container_class']
        )

        print(f"\n✅ Trovati {len(containers)} prodotti")

        products = []

        for idx, container in enumerate(containers, 1):
            try:
                name = container.get_text(strip=True)
                price_elem = container.find(class_=re.compile(r'price', re.I))

                price = None
                if price_elem:
                    price = re.sub(r"[^\d,\.]", "", price_elem.get_text()).replace(",", ".")

                products.append({
                    "Brand": "Diadora",
                    "Modello": name,
                    "Prezzo": price,
                    "Data Rilevazione": datetime.now().strftime("%Y-%m-%d")
                })

            except Exception as e:
                print(f"⚠️ Errore prodotto {idx}: {e}")

        self.all_products = products
        return products

    def create_analysis_report(self, filename="Diadora_Competitive_Analysis.xlsx"):
        if not self.all_products:
            print("❌ Nessun prodotto da analizzare")
            return

        df = pd.DataFrame(self.all_products)

        df['Prezzo'] = pd.to_numeric(df['Prezzo'], errors='coerce')

        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name="Dati Prodotti", index=False)

        print(f"📊 Report salvato in {filename}")

    def run_complete_analysis(self, max_products=200):
        try:
            self.navigate_and_prepare()
            self.auto_detect_selectors()
            self.scrape_products(max_products)
            self.create_analysis_report()
        finally:
            self.close()

    def close(self):
        if self.driver:
            print("\n🔒 Chiusura browser...")
            self.driver.quit()
            print("✅ Browser chiuso\n")


def main():
    analyzer = DiadoraCompetitiveAnalyzer(headless=False, auto_detect=True)
    analyzer.run_complete_analysis(max_products=200)


if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║         DIADORA COMPETITIVE ANALYSIS - VERSIONE INTEGRATA v3.0               ║
║                                                                              ║
║              Auto-Detection Selettori + Analisi Completa                     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝


🔧 CONFIGURAZIONE CHROME WEBDRIVER
👁️  Modalità: Visibile (vedrai il browser)
✅ WebDriver configurato con successo!

📍 NAVIGAZIONE SITO DIADORA

1️⃣ Apertura homepage...
2️⃣ Gestione cookie consent...
   ⚠️ Cookie banner non trovato
3️⃣ Navigazione sezione scarpe...
   🔍 Ricerca link 'Scarpe'...

   ⚠️ Link 'Scarpe' non trovato automaticamente
   👉 Clicca manualmente su 'SCARPE' nel browser


   Premi INVIO per continuare...  


4️⃣ Caricamento prodotti (scroll)...
   ✅ Scroll completato

✅ Pagina pronta per l'analisi!

🔍 AUTO-DETECTION SELETTORI CSS
💾 HTML salvato in diadora_analysis.html
   ✅ Container rilevato: div.product

📦 SCRAPING PRODOTTI DIADORA

✅ Trovati 30 prodotti
📊 Report salvato in Diadora_Competitive_Analysis.xlsx

🔒 Chiusura browser...
✅ Browser chiuso

